In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
df_features = pd.read_csv("../data/processed/cleaned_waitlist.csv")
df_features.head() 

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,...,WL_ID_CODE,PREV_TX,DIAG_KI,MULTIORG,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,53,2018-03-30,...,1526706,Unknown,-1.0,N,13609,died,1,0,0,287.0
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,56,2017-08-16,...,1519605,Unknown,-1.0,N,6975,removed_administrative,0,0,0,2322.0
2,N,NaN,F,O,32.85,2070.0,4099,0.0,47,Not on dialysis,...,1535094,Unknown,-1.0,N,19716,died,1,0,0,604.0
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,61,2019-01-05,...,1527782,Unknown,-1.0,N,8587,removed_too_sick,1,0,0,909.0
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,61,2019-01-10,...,1517273,Unknown,-1.0,N,18352,removed_too_sick,1,0,0,231.0


In [3]:
# Select only the recommended features per data_dictionary.md
feature_cols = [
    'ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR',
    'FUNC_STAT_TCR', 'INIT_STAT', 'INIT_CPRA', 'INIT_AGE',
    'DIALYSIS_DATE', 'INIT_DATE', 'ETHCAT', 'REGION'
]

df_model = df_features[feature_cols].copy()
df_model.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,INIT_DATE,ETHCAT,REGION
0,Y,NaN,F,B,31.63,2080.0,4099,0.0,53,2018-03-30,2020-03-25,2,8
1,Y,NaN,M,A,30.04,2070.0,4099,0.0,56,2017-08-16,2020-02-14,1,5
2,N,NaN,F,O,32.85,2070.0,4099,0.0,47,Not on dialysis,2020-05-27,1,11
3,Y,NaN,M,A,20.00,2090.0,4099,0.0,61,2019-01-05,2020-04-02,5,7
4,Y,NaN,M,AB,23.30,2070.0,4010,0.0,61,2019-01-10,2020-02-05,2,7


In [4]:
categorical_cols = [
    'ON_DIALYSIS', 'GENDER', 'ABO', 'A2A2B_ELIGIBILITY',
    'INIT_STAT', 'ETHCAT', 'REGION'
]

numerical_cols = ['BMI_TCR', 'INIT_CPRA', 'INIT_AGE']

# FUNC_STAT_TCR needs special decoding (per data_dictionary.md) not simple encoding or scaling
# DIALYSIS_DATE / INIT_DATE need date handling (per data_dictionary.md) not simple encoding or scaling

In [5]:
#Caps BMI at a maximum of 80, there was a data entry error
df_model.loc[df_model['BMI_TCR'] > 80, 'BMI_TCR'] = df_model['BMI_TCR'].median()
df_model['BMI_TCR'].describe()

count    494803.000000
mean         28.823687
std           5.809189
min           0.180000
25%          24.660000
50%          28.530000
75%          32.820000
max          78.980000
Name: BMI_TCR, dtype: float64

In [6]:
#encode categorical data
new_df = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
new_df.head()

,BMI_TCR,FUNC_STAT_TCR,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,INIT_DATE,ON_DIALYSIS_Y,GENDER_M,ABO_A1,ABO_A1B,...,REGION_2,REGION_3,REGION_4,REGION_5,REGION_6,REGION_7,REGION_8,REGION_9,REGION_10,REGION_11
0,31.63,2080.0,0.0,53,2018-03-30,2020-03-25,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
1,30.04,2070.0,0.0,56,2017-08-16,2020-02-14,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
2,32.85,2070.0,0.0,47,Not on dialysis,2020-05-27,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,20.00,2090.0,0.0,61,2019-01-05,2020-04-02,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
4,23.30,2070.0,0.0,61,2019-01-10,2020-02-05,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False


In [7]:
#Check INIT_CPRA range/outliers and most frequent values (which is 0.00)
df_model['INIT_CPRA'].describe()
df_model['INIT_CPRA'].value_counts().head(10)

INIT_CPRA
0.00      420160
0.03         904
16.56        586
0.26         583
100.00       568
99.99        542
56.09        500
2.32         499
50.01        496
0.02         474
Name: count, dtype: int64

In [8]:
#normalize numerical data
scaler = StandardScaler()
new_df[numerical_cols] = scaler.fit_transform(new_df[numerical_cols])
new_df[numerical_cols].head()

,BMI_TCR,INIT_CPRA,INIT_AGE
0,0.483082,-0.32562,0.077090
1,0.209378,-0.32562,0.282551
2,0.693094,-0.32562,-0.333834
3,-1.518920,-0.32562,0.624987
4,-0.950854,-0.32562,0.624987


# Decoding FUNC_STAT_TCR

In [9]:
#Decoding FUNC_STAT_TCR
# searching for quantity of each values 
df_model['FUNC_STAT_TCR'].value_counts(dropna=False).sort_index()

FUNC_STAT_TCR
-1.0         3974
 1.0            1
 996.0        121
 998.0       8697
 2010.0       371
 2020.0      2936
 2030.0      1905
 2040.0      8364
 2050.0     23920
 2060.0     36961
 2070.0    111480
 2080.0    139465
 2090.0    109306
 2100.0     35402
 4010.0        43
 4020.0        23
 4030.0        89
 4040.0       187
 4050.0       157
 4060.0       505
 4070.0      1029
 4080.0      2812
 4090.0      2413
 4100.0      4642
Name: count, dtype: int64

In [10]:
#creating a dictionaries for funtionalaity scale & creating new columns
scale_mapping_dict = {}
percent_mapping_dict = {}

scale_mapping_dict[-1] = 'missing'
percent_mapping_dict[-1] = None

scale_mapping_dict[1] = 'adl_independent'
percent_mapping_dict[1] = None

scale_mapping_dict[996] = 'not_applicable'
percent_mapping_dict[996] = None

scale_mapping_dict[998] = 'unknown'
percent_mapping_dict[998] = None

#adult karnofsky: 2010-2100
for x in range(2010,2101,10):
    scale_mapping_dict[x] = 'adult_karnofsky'
    percent_mapping_dict[x] = x-2000

#pediatric Lanksy
for x in range(4010, 4101,10):
    scale_mapping_dict[x] = 'pediatric_lansky'
    percent_mapping_dict[x] = x -4000

df_model['functional_scale'] = df_model['FUNC_STAT_TCR'].map(scale_mapping_dict)
df_model['functional_percent'] = df_model['FUNC_STAT_TCR'].map(percent_mapping_dict)

df_model['functional_scale'] = df_model['functional_scale'].fillna('other/unresolved')

In [11]:
df_model[['FUNC_STAT_TCR', 'functional_scale', 'functional_percent']].head(10)

,FUNC_STAT_TCR,functional_scale,functional_percent
0,2080.0,adult_karnofsky,80.0
1,2070.0,adult_karnofsky,70.0
2,2070.0,adult_karnofsky,70.0
3,2090.0,adult_karnofsky,90.0
4,2070.0,adult_karnofsky,70.0
5,2090.0,adult_karnofsky,90.0
6,2090.0,adult_karnofsky,90.0
7,2070.0,adult_karnofsky,70.0
8,2080.0,adult_karnofsky,80.0
9,2080.0,adult_karnofsky,80.0


In [25]:
df_model['functional_scale'].value_counts()

functional_scale
adult_karnofsky     470110
pediatric_lansky     11900
unknown               8697
missing               3974
not_applicable         121
adl_independent          1
Name: count, dtype: int64

In [12]:
#filling missing values
df_model['functional_percent'] = df_model['functional_percent'].fillna(df_model['functional_percent'].median())
df_model['functional_percent'].isna().sum()

np.int64(0)

In [13]:
#dropping the old columns FUNC_STAT_TCR
df_model = df_model.drop(columns=['FUNC_STAT_TCR'])
df_model.columns.tolist()

['ON_DIALYSIS',
 'A2A2B_ELIGIBILITY',
 'GENDER',
 'ABO',
 'BMI_TCR',
 'INIT_STAT',
 'INIT_CPRA',
 'INIT_AGE',
 'DIALYSIS_DATE',
 'INIT_DATE',
 'ETHCAT',
 'REGION',
 'functional_scale',
 'functional_percent']

In [14]:
#updating col list to include functional_scale and functional_percent
categorical_cols = [
    'ON_DIALYSIS', 'GENDER', 'ABO', 'A2A2B_ELIGIBILITY',
    'INIT_STAT', 'ETHCAT', 'REGION', 'functional_scale'
]

numerical_cols = ['BMI_TCR', 'INIT_CPRA', 'INIT_AGE', 'functional_percent']

In [15]:
#encoding and normalizing new columns
new_df = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

scaler = StandardScaler()
new_df[numerical_cols] = scaler.fit_transform(new_df[numerical_cols])

new_df.head()

,BMI_TCR,INIT_CPRA,INIT_AGE,DIALYSIS_DATE,INIT_DATE,functional_percent,ON_DIALYSIS_Y,GENDER_M,ABO_A1,ABO_A1B,...,REGION_7,REGION_8,REGION_9,REGION_10,REGION_11,functional_scale_adult_karnofsky,functional_scale_missing,functional_scale_not_applicable,functional_scale_pediatric_lansky,functional_scale_unknown
0,0.483082,-0.32562,0.077090,2018-03-30,2020-03-25,0.185972,True,False,False,False,...,False,True,False,False,False,True,False,False,False,False
1,0.209378,-0.32562,0.282551,2017-08-16,2020-02-14,-0.507516,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
2,0.693094,-0.32562,-0.333834,Not on dialysis,2020-05-27,-0.507516,False,False,False,False,...,False,False,False,False,True,True,False,False,False,False
3,-1.518920,-0.32562,0.624987,2019-01-05,2020-04-02,0.879461,True,True,False,False,...,True,False,False,False,False,True,False,False,False,False
4,-0.950854,-0.32562,0.624987,2019-01-10,2020-02-05,-0.507516,True,True,False,False,...,True,False,False,False,False,True,False,False,False,False


# dialysis-duration calculation

In [16]:
#converting init_date and dialysis_date to datetime format; want to find the amount of time patient been on dialysis before listed for transplant
df_model['INIT_DATE'] = pd.to_datetime(df_model['INIT_DATE'])
df_model['DIALYSIS_DATE_cleaned'] = pd.to_datetime(df_model['DIALYSIS_DATE'], errors='coerce') #turns anything in wrong format into missing data 
df_model['DIALYSIS_DATE_cleaned'].isna().sum()

#134235 individuals who have no dialysis data (possibly bc they were never on it)

np.int64(134235)

In [17]:
df_model['dialysis_duration_days'] = (df_model['INIT_DATE'] - df_model['DIALYSIS_DATE_cleaned']).dt.days

df_model['dialysis_duration_days'].describe()

#360568 have a duration of dialysis to waitlist
#min= -3792 started diaylsis after waitlisted for a transplant

count    360568.000000
mean        754.682193
std        1042.641400
min       -3792.000000
25%         181.000000
50%         467.000000
75%        1046.000000
max       15356.000000
Name: dialysis_duration_days, dtype: float64

In [18]:
#fill the individuals with no dialysis date with 0, bc they have zero days on dialysis
df_model['dialysis_duration_days'] = df_model['dialysis_duration_days']. fillna(0)
df_model['dialysis_duration_days'].describe()

count    494803.000000
mean        549.944622
std         951.196943
min       -3792.000000
25%           0.000000
50%         247.000000
75%         762.000000
max       15356.000000
Name: dialysis_duration_days, dtype: float64

In [19]:
# the individuals who started dialysis after being listed (negative durations) is set to 0
# we are answering 'how many days had this patient been on dialysis before listing', which is 0
df_model.loc[df_model['dialysis_duration_days'] < 0, 'dialysis_duration_days'] = 0

df_model['dialysis_duration_days'].describe()

count    494803.000000
mean        582.210546
std         918.094631
min           0.000000
25%           0.000000
50%         247.000000
75%         762.000000
max       15356.000000
Name: dialysis_duration_days, dtype: float64

In [20]:
(df_model['dialysis_duration_days'] < 0).sum()

np.int64(0)

In [ ]:
#update our numerical_cols to include dialysis_duration_days
numerical_cols = ['BMI_TCR', 'INIT_CPRA', 'INIT_AGE', 'functional_percent', 'dialysis_duration_days']

In [22]:
# drop the raw date columns now that dialysis_duration_days is calculated
df_model = df_model.drop(columns=['DIALYSIS_DATE', 'DIALYSIS_DATE_cleaned', 'INIT_DATE'])
df_model.columns.tolist()

['ON_DIALYSIS',
 'A2A2B_ELIGIBILITY',
 'GENDER',
 'ABO',
 'BMI_TCR',
 'INIT_STAT',
 'INIT_CPRA',
 'INIT_AGE',
 'ETHCAT',
 'REGION',
 'functional_scale',
 'functional_percent',
 'dialysis_duration_days']

In [ ]:
#final encoding and normalizing, including dialysis_duration_days
new_df= pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

scaler= StandardScaler()
new_df[numerical_cols] = scaler.fit_transform(new_df[numerical_cols])

new_df.head()

,BMI_TCR,INIT_CPRA,INIT_AGE,functional_percent,dialysis_duration_days,ON_DIALYSIS_Y,GENDER_M,ABO_A1,ABO_A1B,ABO_A2,...,REGION_7,REGION_8,REGION_9,REGION_10,REGION_11,functional_scale_adult_karnofsky,functional_scale_missing,functional_scale_not_applicable,functional_scale_pediatric_lansky,functional_scale_unknown
0,0.483082,-0.32562,0.077090,0.185972,0.156617,True,False,False,False,False,...,False,True,False,False,False,True,False,False,False,False
1,0.209378,-0.32562,0.282551,-0.507516,0.359211,True,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,0.693094,-0.32562,-0.333834,-0.507516,-0.634152,False,False,False,False,False,...,False,False,False,False,True,True,False,False,False,False
3,-1.518920,-0.32562,0.624987,0.879461,-0.140738,True,True,False,False,False,...,True,False,False,False,False,True,False,False,False,False
4,-0.950854,-0.32562,0.624987,-0.507516,-0.208269,True,True,False,False,False,...,True,False,False,False,False,True,False,False,False,False


In [26]:
new_df.shape

(494803, 41)

In [ ]:
new_df.isna().sum().sum() 


np.int64(0)